In [ ]:
# ============================================================
# Phase-distance benchmark on simulated vector fields
# Single 4x8 figure + one combined table
# ============================================================

%load_ext autoreload
%autoreload 2

import sys
import os

PROJECT_ROOT = os.path.abspath("..")

if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scripts.TPS import ThinPlateSpline
from scripts.plotting import *
from scripts.VectorFieldEmbedder import *
from scripts.evaluation import evaluate_embedding_method
from flowmap.plot import plot_velocity_stream

# ============================================================
# load vector fields
# ============================================================

def load_vector_field(csv_path):

    df = pd.read_csv(csv_path)

    X = df[["x", "y"]].values
    V = df[["vx", "vy"]].values
    time = df["time"].values

    return X, V, time

# ============================================================
# datasets
# ============================================================

path_map = {
    "straight_line": "./data/1d/straight_line.csv",
    "sine_curve": "./data/1d/sine_curve.csv",
    "branch_2": "./data/1d/branch_2.csv",
    "branch_4": "./data/1d/branch_4.csv",
    "rotation": "./data/2d/rotation.csv",
    "spiral": "./data/2d/spiral.csv",
    "saddle": "./data/2d/saddle.csv",
    "quadratic_source_sink": "./data/2d/quadratic_source_sink.csv",
}

plot_order = [
    "straight_line",
    "sine_curve",
    "branch_2",
    "branch_4",
    "rotation",
    "spiral",
    "saddle",
    "quadratic_source_sink",
]

# ============================================================
# simulate noisy data ONCE
# ============================================================

np.random.seed(42)

simulation_results = {}

noise = 0.3
extra_dim = 5

for name, path in path_map.items():

    X_gt, V_gt, time = load_vector_field(path)

    X_noisy = X_gt + np.random.normal(
        scale=noise,
        size=X_gt.shape,
    )

    V_noisy = V_gt + np.random.normal(
        scale=noise,
        size=V_gt.shape,
    )

    X_dummy = np.random.normal(
        scale=noise,
        size=(X_gt.shape[0], extra_dim),
    )

    V_dummy = np.random.normal(
        scale=noise,
        size=(V_gt.shape[0], extra_dim),
    )

    X = np.hstack([X_noisy, X_dummy])
    V = np.hstack([V_noisy, V_dummy])

    simulation_results[name] = dict(
        X=X,
        V=V,
        X_gt=X_gt,
        V_gt=V_gt,
        true_time=time,
    )

# ============================================================
# benchmark settings
# ============================================================

alphas = [0.0, 0.5, 1.0, 2.0]

fig_dir = "./figures/simulation_phase"
os.makedirs(fig_dir, exist_ok=True)

all_scores = []

# ============================================================
# giant figure
# ============================================================

fig, axs = plt.subplots(
    len(alphas),
    len(plot_order),
    figsize=(32, 16),
)

# ============================================================
# benchmark loop
# ============================================================

for row_idx, alpha in enumerate(alphas):

    print("\n" + "=" * 80)
    print(f"Running alpha = {alpha}")
    print("=" * 80)

    embedding_results = {}

    # --------------------------------------------------------
    # fit embeddings
    # --------------------------------------------------------

    for name, data in simulation_results.items():

        X = data["X"]
        V = data["V"]

        time = data["true_time"]

        time = (
            time - np.min(time)
        ) / (
            np.max(time) - np.min(time) + 1e-12
        )

        emb = VectorFieldEmbedder(
            X,
            V,
            alpha=alpha,
            use_PCA=False,
            dist_method="phase",
            embed_kwargs={
                "n_neighbors": 20,
                "min_dist": 0.4,
            },
        )

        emb.initialize_embedding()

        embedding_results[name] = {
            "embedder": emb,
            "time": time,
        }

    # --------------------------------------------------------
    # plotting
    # --------------------------------------------------------

    for col_idx, name in enumerate(plot_order):

        ax = axs[row_idx, col_idx]

        emb = embedding_results[name]["embedder"]

        time = embedding_results[name]["time"]

        X_gt = simulation_results[name]["X_gt"]
        V_gt = simulation_results[name]["V_gt"]

        X_emb = emb.X_emb
        V_emb = emb.tps_vf.predict(X_emb)

        # ----------------------------------------------------
        # layout tuning
        # ----------------------------------------------------

        if name == "straight_line":
            stream_density, aspect = 0.4, 2.5

        elif name == "sine_curve":
            stream_density, aspect = 0.5, 2.0

        elif name == "branch_2":
            stream_density, aspect = 0.5, 1.5

        elif name == "branch_4":
            stream_density, aspect = 0.8, 1.5

        else:
            stream_density, aspect = 0.6, "equal"

        # if name == "straight_line":
        #     stream_density, aspect = 0., 2.5

        # elif name == "sine_curve":
        #     stream_density, aspect = 0., 2.0

        # elif name == "branch_2":
        #     stream_density, aspect = 0., 1.5

        # elif name == "branch_4":
        #     stream_density, aspect = 0., 1.5

        # else:
        #     stream_density, aspect = 0., "equal"

        # ----------------------------------------------------
        # plot
        # ----------------------------------------------------

        plot_velocity_stream(
            X_2d=X_emb,
            spline=emb.tps_vf,
            grid_density=1.1,
            stream_density=stream_density,
            scatter_color=time,
            scatter_size=200,
            scatter_alpha=0.1,
            ax=ax,
            title=None,
            aspect=aspect,
            vmin=0.0,
            vmax=1.0,
            arrowsize=3.0,
            cmap="viridis",
            show_axes=False,
            streamline_thickness=4.0,
            grid_size=50,
            pad_frac=0.1,
        )

        # ----------------------------------------------------
        # titles
        # ----------------------------------------------------

        if row_idx == 0:
            ax.set_title(
                name.replace("_", "\n"),
                fontsize=11,
            )

        if col_idx == 0:
            ax.set_ylabel(
                rf"$\alpha={alpha}$",
                fontsize=16,
                rotation=90,
            )

        # ----------------------------------------------------
        # evaluate
        # ----------------------------------------------------

        metrics = evaluate_embedding_method(
            X_gt,
            X_emb,
            V_gt,
            V_emb,
            k=30,
        )

        metrics["dataset"] = name
        metrics["alpha"] = alpha

        all_scores.append(metrics)

# ============================================================
# finalize figure
# ============================================================

plt.tight_layout()

fig_path = os.path.join(
    fig_dir,
    "phase_distance_benchmark_grid.png",
)

plt.savefig(
    fig_path,
    dpi=300,
    bbox_inches="tight",
)

print(f"\nSaved figure to: {fig_path}")

plt.show()

# ============================================================
# giant evaluation table
# ============================================================

df_all = pd.DataFrame(all_scores)

metric_cols = [
    c for c in df_all.columns
    if c not in ["dataset", "alpha"]
]

# ============================================================
# rounded display
# ============================================================

display_df = df_all.copy()

for c in metric_cols:
    display_df[c] = display_df[c].round(4)

display_df = display_df.sort_values(
    ["alpha", "dataset"]
)

display(display_df)

# ============================================================
# save combined table
# ============================================================

combined_csv = os.path.join(
    fig_dir,
    "phase_distance_benchmark_all_scores.csv",
)

display_df.to_csv(
    combined_csv,
    index=False,
)

print(f"\nSaved combined scores to: {combined_csv}")

# ============================================================
# mean metrics by alpha
# ============================================================

mean_df = (
    df_all
    .groupby("alpha")[metric_cols]
    .mean()
    .round(4)
)

print("\nMean metrics by alpha:")
display(mean_df)

# ============================================================
# save mean summary
# ============================================================

mean_csv = os.path.join(
    fig_dir,
    "phase_distance_benchmark_mean_scores.csv",
)

mean_df.to_csv(mean_csv)

print(f"Saved mean scores to: {mean_csv}")

# ============================================================
# save latex table
# ============================================================

latex_path = os.path.join(
    fig_dir,
    "phase_distance_benchmark_mean_table.tex",
)

with open(latex_path, "w") as f:
    f.write(mean_df.to_latex())

print(f"Saved LaTeX table to: {latex_path}")